# 4. Machine Learning
End-to-end model training, evaluation, and inference workflow.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split

from src.config.core import config
from src.predict import make_prediction
from src.processing.data_manager import load_dataset, load_pipeline, model_file_name
from src.train_pipeline import run_training

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent


In [ ]:
train_metrics = run_training()
print('Training metrics:', train_metrics)

model = load_pipeline(model_file_name())
print('Loaded model artifact:', model_file_name())


In [ ]:
data = load_dataset(config.app_config.training_data_file)
x = data.drop(columns=[config.ml_config.target])
y = data[config.ml_config.target].astype(int)

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y,
    test_size=config.ml_config.test_size,
    random_state=config.ml_config.random_state,
    stratify=y,
)

pred = model.predict(x_valid)
proba = model.predict_proba(x_valid)[:, 1]


In [ ]:
print('Validation Accuracy:', round(accuracy_score(y_valid, pred), 4))
print('Validation ROC AUC :', round(roc_auc_score(y_valid, proba), 4))
print(classification_report(y_valid, pred))


In [ ]:
precision, recall, thresholds = precision_recall_curve(y_valid, proba)
threshold_frame = pd.DataFrame({
    'threshold': list(thresholds) + [1.0],
    'precision': precision,
    'recall': recall,
})
threshold_frame.head(10)


In [ ]:
test_df = pd.read_csv(ROOT / 'src/data/test.csv')
test_predictions = make_prediction(test_df)

pred_df = pd.DataFrame({
    'enrollee_id': test_df['enrollee_id'],
    'prediction': test_predictions['predictions'],
})
if 'probabilities' in test_predictions:
    pred_df['probability'] = test_predictions['probabilities']

pred_df.head()


In [ ]:
output_dir = ROOT / 'notebooks/outputs'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'test_predictions.csv'
pred_df.to_csv(output_path, index=False)
print('Saved predictions to:', output_path)


In [ ]:
api_like_payload = {
    'inputs': [
        {
            'city': 'city_41',
            'city_development_index': 0.827,
            'gender': 'Male',
            'relevent_experience': 'Has relevent experience',
            'enrolled_university': 'Full time course',
            'education_level': 'Graduate',
            'major_discipline': 'STEM',
            'experience': '9',
            'company_size': '<10',
            'company_type': 'Pvt Ltd',
            'last_new_job': '1',
            'training_hours': 21.0
        }
    ]
}

make_prediction(pd.DataFrame(api_like_payload['inputs']))


This notebook can be reused for regression checks after pipeline updates: retrain -> evaluate -> run batch inference -> export predictions.